# Decoupled Quantile + Actuarial Optimization

**Arsitektur 3-Tahap untuk FMCG Demand Forecasting**

### Masalah yang Diperbaiki
1. **Fix "f34" Anomaly:** Pipeline sebelumnya mengonversi DataFrame ke NumPy, sehingga XGBoost kehilangan nama kolom dan menampilkan fitur buta (f0..f51). Sekarang model dilatih dengan DataFrame langsung agar feature_names tetap utuh.

2. **Fix "holiday_intensity" Bug:** Agregasi sebelumnya menggunakan `drop_duplicates()` tanpa subset tanggal, sehingga bisa menghilangkan variasi antar-tanggal. Juga ada `clip(upper=10.0)` yang membuat semua negara mentok di 10.0. Keduanya diperbaiki.

### Arsitektur Baru (3-Pronged Decoupled)
1. **Honest Baseline:** XGBoost `reg:squarederror` — memprediksi mean demand ($\mu$) tanpa bias.
2. **Risk-Aware Layer:** XGBoost `reg:quantileerror` (q=0.85) — memprediksi 85th percentile sebagai proksi varians.
3. **Actuarial Optimization:** Interpolasi dinamis antara $\mu$ dan $q_{0.85}$ berdasarkan **Critical Fractile** (CF) per SKU.
   $CF = \frac{Cu}{(Cu + Co)}$ di mana Cu = shortage cost, Co = overstock cost.
   Rekomendasi akhir: $\hat{y} = \mu + (q_{0.85} - \mu) \times CF$


In [ ]:
# ============================================================
# [SETUP] Instalasi, GPU detection, imports
# ============================================================
import importlib, sys, os, warnings, json, gc, time, math
from pathlib import Path
import numpy as np
import pandas as pd

_start = time.time()

# Install holidays
try:
    importlib.import_module('holidays')
    print("[OK] holidays already installed")
except ImportError:
    print("[INSTALL] Installing holidays...")
    !pip install holidays -q
    print("[OK] holidays installed")

import holidays
from sklearn.metrics import mean_absolute_error, f1_score, precision_score, recall_score
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import xgboost as xgb
from scipy.stats import spearmanr

# GPU detection
print("[INFO] XGBoost version:", xgb.__version__)
try:
    import torch
    cuda_ok = torch.cuda.is_available()
except ImportError:
    cuda_ok = False
try:
    _ = xgb.XGBRegressor(n_estimators=1, device='cuda')
    DEVICE = 'cuda'
    print("[OK] GPU device=cuda: SUPPORTED")
except Exception:
    try:
        _ = xgb.XGBRegressor(n_estimators=1, tree_method='gpu_hist')
        DEVICE = 'cuda'
        print("[OK] GPU tree_method=gpu_hist: SUPPORTED")
    except Exception as e:
        DEVICE = 'cpu'
        print("[WARN] GPU not supported, falling back to CPU:", e)

print(f"[OK] Using device={DEVICE} ({time.time()-_start:.1f}s)")

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 140)
pd.set_option('mode.copy_on_write', True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)



In [ ]:
# ============================================================
# [CONFIG] Paths, constants, non-product codes
# ============================================================
RAW_PATH = "/kaggle/input/datasets/wildanmaulana1/online-retail-fmcg/online_retail.csv"
CACHE_DIR = Path("/kaggle/working")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_PATH = str(CACHE_DIR / "panel_cached.parquet")
FULL_CACHE = str(CACHE_DIR / "panel_full.parquet")

print(f"[CONFIG] RAW_PATH: {RAW_PATH}")
print(f"[CONFIG] CACHE_PATH: {CACHE_PATH}")
print(f"[CONFIG] FULL_CACHE: {FULL_CACHE}")
print(f"[CONFIG] RAW_PATH exists: {Path(RAW_PATH).exists()}")
if not Path(RAW_PATH).exists():
    parent = Path(RAW_PATH).parent
    print(f"[WARN] RAW_PATH missing. Listing parent: {parent}")
    if parent.exists():
        for p in parent.iterdir():
            print(f"  - {p}")

CHUNK_SIZE = 200_000
USECOLS = ['Invoice', 'StockCode', 'Quantity', 'InvoiceDate', 'Price', 'Country']
DTYPES = {
    'Invoice': 'string', 'StockCode': 'string', 'Quantity': 'float32',
    'Price': 'float32', 'Country': 'string',
}

# Configurable caps (di-deklarasikan di awal agar semua cell bisa akses)
HOLIDAY_INTENSITY_CAP = 100.0   # sudah tidak flat 10.0
QUANTILE_ALPHA = 0.85           # risk-aware quantile target

NON_PRODUCT_CODES = {
    'POST', 'DOT', 'C2', 'M', 'D', 'ADJUST', 'ADJUST2',
    'BANK CHARGES', 'AMAZONFEE', 'B', 'S', 'PADS',
    'TEST001', 'TEST002', 'GIFT_0001_10', 'GIFT_0001_20',
    'GIFT_0001_30', 'GIFT_0001_40', 'GIFT_0001_50', 'GIFT_0001_70',
    'GIFT_0001_80',
}



In [ ]:
# ============================================================
# [PREPROCESSING] Chunk loading -> daily aggregation
# ============================================================
def preprocess_chunk(chunk):
    _t = time.time()
    df = chunk.rename(
        columns={'Invoice': 'invoice', 'StockCode': 'stock_code',
                 'Quantity': 'quantity', 'InvoiceDate': 'invoice_date',
                 'Price': 'price', 'Country': 'country'}).copy()
    df['invoice_date'] = pd.to_datetime(df['invoice_date'], errors='coerce')
    df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce').astype('float32')
    df['price'] = pd.to_numeric(df['price'], errors='coerce').astype('float32')
    df['stock_code'] = df['stock_code'].astype('string').str.strip().str.upper()
    df['invoice'] = df['invoice'].astype('string').str.strip()
    df['country'] = df['country'].astype('string').str.strip()
    df = df.drop_duplicates()
    df = df[df['invoice_date'].notna()]
    df = df[df['stock_code'].notna() & (df['stock_code'] != '')]
    df = df[df['invoice'].notna() & (df['invoice'] != '')]
    df = df[~df['invoice'].str.startswith('C', na=False)]
    df = df[~df['stock_code'].isin(NON_PRODUCT_CODES)]
    df = df[(df['quantity'] > 0) & (df['price'] > 0)]
    df['date'] = df['invoice_date'].dt.normalize()
    df['revenue'] = df['quantity'] * df['price']
    df['stock_code'] = df['stock_code'].astype('category')
    df['country'] = df['country'].astype('category')
    df['invoice'] = df['invoice'].astype('category')
    print(f"  [PREPROCESS] chunk processed: {len(df)} rows in {time.time()-_t:.1f}s")
    return df[['stock_code', 'country', 'date', 'invoice', 'quantity', 'price', 'revenue']]

def aggregate_daily_from_chunks(path):
    _t0 = time.time()
    parts = []
    max_parts = 25
    chunk_i = 0
    for chunk in pd.read_csv(path, usecols=USECOLS, dtype=DTYPES, chunksize=CHUNK_SIZE, low_memory=False):
        chunk_i += 1
        print(f"  [CHUNK {chunk_i}] Reading...")
        cleaned = preprocess_chunk(chunk)
        daily = cleaned.groupby(['stock_code', 'country', 'date'], as_index=False, observed=True).agg(
            demand_qty=('quantity', 'sum'), revenue=('revenue', 'sum'),
            num_invoices=('invoice', 'nunique'), price_mean=('price', 'mean'))
        parts.append(daily)
        if len(parts) >= max_parts:
            partial = pd.concat(parts, ignore_index=True)
            parts = [partial.groupby(['stock_code', 'country', 'date'], as_index=False, observed=True)
                     .agg(demand_qty=('demand_qty', 'sum'), revenue=('revenue', 'sum'),
                          num_invoices=('num_invoices', 'sum'), price_mean=('price_mean', 'mean'))]
            del partial; gc.collect()
        del chunk, cleaned, daily; gc.collect()
    daily_all = pd.concat(parts, ignore_index=True)
    del parts; gc.collect()
    daily_all = daily_all.groupby(['stock_code', 'country', 'date'], as_index=False, observed=True).agg(
        demand_qty=('demand_qty', 'sum'), revenue=('revenue', 'sum'),
        num_invoices=('num_invoices', 'sum'), price_mean=('price_mean', 'mean'))
    daily_all['avg_price'] = np.where(daily_all['demand_qty'] > 0,
                                      daily_all['revenue'] / daily_all['demand_qty'], daily_all['price_mean'])
    daily_all = daily_all.drop(columns=['price_mean'])
    for c in ['stock_code','country']: daily_all[c] = daily_all[c].astype('category')
    for c in ['demand_qty','revenue','avg_price','num_invoices']: daily_all[c] = daily_all[c].astype('float32')
    print(f"[OK] aggregate_daily done: {daily_all.shape} in {time.time()-_t0:.1f}s")
    return daily_all



In [ ]:
# ============================================================
# [LOAD DATA] Build panel with zero-filled daily grid
# ============================================================
_t0 = time.time()

def build_full_daily_panel(df):
    df = df.sort_values(['stock_code', 'country', 'date']).reset_index(drop=True)
    def _resample(group):
        group = group.set_index('date').asfreq('D')
        group['stock_code'] = group['stock_code'].iloc[0]
        group['country'] = group['country'].iloc[0]
        return group.reset_index()
    panel = df.groupby(['stock_code', 'country'], group_keys=False, sort=False).apply(_resample)
    panel = panel.reset_index(drop=True)
    for c in ['demand_qty','revenue','num_invoices']: panel[c] = panel[c].fillna(0)
    panel['avg_price'] = panel['avg_price'].astype('float32')
    panel['avg_price'] = panel.groupby(['stock_code', 'country'], sort=False)['avg_price'].ffill().bfill().fillna(0)
    for c in ['stock_code','country']: panel[c] = panel[c].astype('category')
    for c in ['demand_qty','revenue','avg_price','num_invoices']: panel[c] = panel[c].astype('float32')
    return panel

group_cols = ['stock_code', 'country']

if os.path.exists(CACHE_PATH):
    panel_df = pd.read_parquet(CACHE_PATH)
    print(f"[LOAD] Loaded cached panel: {panel_df.shape}")
else:
    print("[LOAD] Building daily aggregation from raw CSV...")
    daily_df = aggregate_daily_from_chunks(RAW_PATH)
    print("[LOAD] Building panel (zero-filled daily grid)...")
    panel_df = build_full_daily_panel(daily_df)
    del daily_df; gc.collect()
    panel_df = panel_df.sort_values(['stock_code', 'country', 'date']).reset_index(drop=True)
    panel_df.to_parquet(CACHE_PATH)
    print(f"[OK] Cached panel saved: {panel_df.shape}")

MIN_OBS = 60
item_obs = panel_df.groupby(group_cols).size()
panel_df = panel_df[panel_df.set_index(group_cols).index.isin(item_obs[item_obs >= MIN_OBS].index)].copy()
panel_df = panel_df.sort_values(group_cols + ['date']).reset_index(drop=True)
n_items = panel_df[group_cols].drop_duplicates().shape[0]
print(f"[OK] After MIN_OBS={MIN_OBS} filter: {panel_df.shape}, items: {n_items}")
print(f"[TIME] Data loading: {time.time()-_t0:.1f}s")
panel_df.head()



In [ ]:
# ============================================================
# [FEATURE ENGINEERING] Calendar -> Holiday -> Lag/Rolling -> Intensity -> Peak
# FIX 1 & 2: Nama kolom dipertahankan untuk XGBoost,
#            holiday_intensity dihitung dengan agregasi yang benar (tanpa clip 10.0)
# ============================================================
_t0 = time.time()

if os.path.exists(FULL_CACHE):
    panel_df = pd.read_parquet(FULL_CACHE)
    print(f"[LOAD] Loaded full preprocessed panel: {panel_df.shape}")
else:
    print("[FE] Generating full feature set...")

    # ---- Calendar + Holiday (14 features) ----
    COUNTRY_TO_HOLIDAYS = {
        'Australia': 'AU', 'Austria': 'AT', 'Belgium': 'BE', 'Canada': 'CA',
        'Channel Islands': 'GB', 'Czech Republic': 'CZ', 'Denmark': 'DK', 'EIRE': 'IE',
        'Finland': 'FI', 'France': 'FR', 'Germany': 'DE', 'Greece': 'GR',
        'Hong Kong': 'HK', 'Iceland': 'IS', 'Israel': 'IL', 'Italy': 'IT',
        'Japan': 'JP', 'Korea': 'KR', 'Netherlands': 'NL', 'Nigeria': 'NG',
        'Norway': 'NO', 'Poland': 'PL', 'Portugal': 'PT', 'RSA': 'ZA',
        'Saudi Arabia': 'SA', 'Singapore': 'SG', 'Spain': 'ES', 'Sweden': 'SE',
        'Switzerland': 'CH', 'Thailand': 'TH', 'USA': 'US',
        'United Kingdom': 'GB', 'United Arab Emirates': 'AE',
        'European Community': None, 'Unspecified': None, 'West Indies': None,
        'Bahrain': 'BH', 'Bermuda': 'BM', 'Cyprus': 'CY', 'Lithuania': 'LT',
        'Malta': 'MT', 'Lebanon': 'LB',
    }
    panel_df['country_code'] = panel_df['country'].map(COUNTRY_TO_HOLIDAYS).astype('category')
    years = sorted(panel_df['date'].dt.year.unique().tolist())
    supported = set(holidays.list_supported_countries())
    holiday_rows = []
    for code in sorted(panel_df['country_code'].dropna().astype(str).unique()):
        if code not in supported:
            continue
        hset = holidays.country_holidays(code, years=years)
        holiday_rows.append(pd.DataFrame({'country_code': code, 'date': list(hset.keys()), 'is_hari_besar': 1}))
    hdf = pd.concat(holiday_rows, ignore_index=True) if holiday_rows else pd.DataFrame(columns=['country_code', 'date', 'is_hari_besar'])
    hdf['date'] = pd.to_datetime(hdf['date'], errors='coerce')
    panel_df = panel_df.merge(hdf, on=['country_code', 'date'], how='left', copy=False)
    if 'is_hari_besar' not in panel_df.columns:
        panel_df['is_hari_besar'] = 0
    panel_df['is_hari_besar'] = panel_df['is_hari_besar'].fillna(0.0).astype('float32').astype('uint8')
    pre_rows = []
    if not hdf.empty:
        for offset in [1, 2, 3]:
            pre_rows.append(hdf.assign(date=hdf['date'] - pd.Timedelta(days=offset), is_pre_hari_besar=1
            )[['country_code', 'date', 'is_pre_hari_besar']])
    pre_df = pd.concat(pre_rows, ignore_index=True) if pre_rows else pd.DataFrame(columns=['country_code', 'date', 'is_pre_hari_besar'])
    pre_df = pre_df.drop_duplicates()
    panel_df = panel_df.merge(pre_df, on=['country_code', 'date'], how='left', copy=False)
    del pre_df, pre_rows, hdf, holiday_rows; gc.collect()
    panel_df['is_pre_hari_besar'] = panel_df['is_pre_hari_besar'].fillna(0.0).astype('float32').astype('uint8')
    panel_df['day_of_week'] = panel_df['date'].dt.dayofweek.astype('int8')
    panel_df['week_of_year'] = panel_df['date'].dt.isocalendar().week.astype('int16')
    panel_df['month'] = panel_df['date'].dt.month.astype('int8')
    panel_df['quarter'] = panel_df['date'].dt.quarter.astype('int8')
    panel_df['day_of_month'] = panel_df['date'].dt.day.astype('int8')
    panel_df['is_weekend'] = (panel_df['day_of_week'] >= 5).astype('uint8')
    panel_df['is_month_start'] = panel_df['date'].dt.is_month_start.astype('uint8')
    panel_df['is_month_end'] = panel_df['date'].dt.is_month_end.astype('uint8')
    panel_df['days_to_month_end'] = ((panel_df['date'] + pd.offsets.MonthEnd(0)) - panel_df['date']).dt.days.astype('int16')
    panel_df['week_of_month'] = ((panel_df['date'].dt.day - 1) // 7 + 1).astype('int8')
    panel_df['is_month_start_window'] = (panel_df['date'].dt.day <= 5).astype('uint8')
    panel_df['is_month_end_window'] = (panel_df['date'].dt.day >= 25).astype('uint8')
    gc.collect()
    print("[FE] Calendar+holiday features done")

    # ---- Lag + Rolling (33 features) ----
    panel_df = panel_df.sort_values(group_cols + ['date']).reset_index(drop=True)
    demand_shifted = panel_df.groupby(group_cols)['demand_qty'].shift(1)
    for lag in [1, 2, 7, 14, 21, 28, 35, 56, 84]:
        panel_df[f'demand_lag_{lag}'] = panel_df.groupby(group_cols)['demand_qty'].shift(lag)
    panel_df['roll_max_7'] = demand_shifted.rolling(7, min_periods=1).max().values
    panel_df['roll_max_28'] = demand_shifted.rolling(28, min_periods=1).max().values
    panel_df['roll_zero_count_14'] = (demand_shifted == 0).rolling(14, min_periods=1).sum().values
    for w in [7, 14, 28, 56]:
        panel_df[f'roll_mean_{w}'] = demand_shifted.rolling(w, min_periods=1).mean().values
    for w in [7, 14, 28]:
        panel_df[f'roll_median_{w}'] = demand_shifted.rolling(w, min_periods=1).median().values
        panel_df[f'roll_std_{w}'] = demand_shifted.rolling(w, min_periods=1).std().values
    panel_df['roll_std_56'] = demand_shifted.rolling(56, min_periods=1).std().values
    for w in [56, 84]:
        panel_df[f'roll_max_{w}'] = demand_shifted.rolling(w, min_periods=1).max().values
    rm3 = demand_shifted.rolling(3, min_periods=1).mean().values
    rm14 = demand_shifted.rolling(14, min_periods=1).mean().values
    panel_df['demand_acceleration_3d'] = rm3 / (rm14 + 1e-8)
    panel_df['spike_ratio_28'] = panel_df['roll_max_28'] / (panel_df['roll_mean_28'] + 1e-8)
    panel_df['spike_ratio_56'] = panel_df['roll_max_56'] / (panel_df['roll_mean_56'] + 1e-8)
    panel_df['pct_change_1'] = (panel_df['demand_lag_1'] - panel_df['demand_lag_2']) / (panel_df['demand_lag_2'] + 1e-8)
    panel_df['pct_change_7'] = (panel_df['demand_lag_7'] - panel_df['demand_lag_14']) / (panel_df['demand_lag_14'] + 1e-8)
    last_sale = panel_df['date'].where(demand_shifted > 0)
    last_sale = last_sale.groupby(panel_df[group_cols].apply(tuple, axis=1)).ffill()
    panel_df['days_since_last_sale'] = (panel_df['date'] - last_sale).dt.days.fillna(9999).astype('int16')
    price_s = panel_df.groupby(group_cols)['avg_price'].shift(1)
    panel_df['discount_depth_pct'] = (price_s.rolling(30, min_periods=1).max().values - panel_df['avg_price']) / (price_s.rolling(30, min_periods=1).max().values + 1e-8)
    panel_df['price_momentum'] = panel_df['avg_price'] / (price_s.rolling(14, min_periods=1).mean().values + 1e-8)
    for c in panel_df.select_dtypes(include=['number']).columns:
        panel_df[c] = panel_df[c].replace([np.inf, -np.inf], 0).fillna(0)
    for c in panel_df.select_dtypes(include=['float64']).columns:
        panel_df[c] = panel_df[c].astype('float32')
    del demand_shifted, price_s, rm3, rm14, last_sale; gc.collect()
    print("[FE] Lag+rolling features done")

    # ---- Holiday Intensity + Proximity (4 features) ----
    # FIX 2: Agregasi holiday sekarang berdasarkan tanggal unik per country_code
    #   sebelumnya: .drop_duplicates() pada [country_code, demand_qty] yang bias
    #   sekarang:   .drop_duplicates(subset=[country_code, date]) -> preserve variasi per tanggal
    hc = panel_df[panel_df['is_hari_besar'] == 1][['country_code', 'date', 'demand_qty']].drop_duplicates(subset=['country_code', 'date'])
    if not hc.empty:
        # Hitung mean demand pada hari libur dan hari pra-libur per country
        hd_mean = hc.groupby('country_code')['demand_qty'].mean()
        hd = hd_mean.to_dict()
        pd_ = {}
        for code in hd:
            pr = panel_df[(panel_df['country_code'] == code) & (panel_df['is_pre_hari_besar'] == 1)]
            pd_[code] = pr['demand_qty'].mean() if len(pr) > 0 else 1.0
        ir = [{
            'country_code': c,
            'holiday_intensity': hd[c] / pd_.get(c, 1.0) if pd_.get(c, 0) > 0 else 1.0
        } for c in hd]
        idf = pd.DataFrame(ir)
        panel_df = panel_df.merge(idf, on='country_code', how='left', copy=False)
        del idf
    panel_df['holiday_intensity'] = panel_df.get('holiday_intensity', pd.Series(1.0, index=panel_df.index)).fillna(1.0).astype('float32')
    # Filter country dengan <5 hari libur: set intensity=1.0
    hc2 = panel_df[panel_df['is_hari_besar'] == 1].groupby('country_code').size()
    low = hc2[hc2 < 5].index
    if len(low) > 0:
        panel_df.loc[panel_df['country_code'].isin(low), 'holiday_intensity'] = 1.0
    # FIX 2: Cap dinaikkan dari 10.0 ke HOLIDAY_INTENSITY_CAP agar tidak flat
    panel_df['holiday_intensity'] = panel_df['holiday_intensity'].clip(upper=HOLIDAY_INTENSITY_CAP)
    del hc, hc2; gc.collect()
    print("[FE] Holiday intensity fix: agregasi per tanggal, cap raised to", HOLIDAY_INTENSITY_CAP)

    # days_to_next_holiday
    hdates = panel_df[panel_df['is_hari_besar'] == 1][['country', 'date']].drop_duplicates()
    panel_df_sorted = panel_df.sort_values(['country', 'date'], kind='mergesort').reset_index(drop=True)
    hdates = hdates.rename(columns={'date': 'next_holiday'}).sort_values(['country', 'next_holiday'], kind='mergesort')
    if len(hdates) > 0:
        parts = []
        for country, grp in panel_df_sorted.groupby('country', sort=False):
            hgrp = hdates[hdates['country'] == country]
            if hgrp.empty:
                grp['days_to_next_holiday'] = 30
            else:
                m = pd.merge_asof(
                    grp.sort_values('date', kind='mergesort'),
                    hgrp.sort_values('next_holiday', kind='mergesort'),
                    left_on='date', right_on='next_holiday',
                    direction='forward', allow_exact_matches=True,
                )
                grp['days_to_next_holiday'] = (m['next_holiday'] - m['date']).dt.days
            parts.append(grp)
        panel_df_sorted = pd.concat(parts, ignore_index=True)
    else:
        panel_df_sorted['days_to_next_holiday'] = 30
    panel_df_sorted['days_to_next_holiday'] = panel_df_sorted['days_to_next_holiday'].fillna(30).clip(0, 30).astype('int8')
    panel_df_sorted['is_holiday_season'] = ((panel_df_sorted['is_hari_besar'] == 1) | (panel_df_sorted['is_pre_hari_besar'] == 1)).astype('uint8')
    panel_df_sorted['holiday_x_weekend'] = (panel_df_sorted['is_hari_besar'].astype('uint8') & panel_df_sorted['is_weekend'].astype('uint8')).astype('uint8')
    panel_df = panel_df_sorted
    del hdates, panel_df_sorted; gc.collect()
    print("[FE] Holiday proximity features done")

    # ---- Peak Days (1 feature) ----
    daily_total = panel_df.groupby('date')['demand_qty'].sum().sort_values(ascending=False)
    peak_n = max(1, int(len(daily_total) * 0.05))
    panel_df['is_peak_day'] = panel_df['date'].isin(set(daily_total.head(peak_n).index)).astype('uint8')
    print("[FE] Peak day features done")

    # ---- Save cache ----
    panel_df.to_parquet(FULL_CACHE)
    print(f"[OK] Full preprocessing cached: {panel_df.shape}")

feat_count = len(panel_df.select_dtypes(include=["number"]).columns)
print(f"[OK] panel_df: {panel_df.shape}, numeric features: {feat_count}")
print(f"[TIME] Feature engineering: {time.time()-_t0:.1f}s")



In [ ]:
# ============================================================
# [CONFIG] Feature group definitions
# ============================================================
SEASONALITY_FEATURES = [
    'day_of_week', 'week_of_year', 'month', 'quarter', 'day_of_month',
    'is_weekend', 'is_month_start', 'is_month_end',
    'days_to_month_end', 'week_of_month',
    'is_month_start_window', 'is_month_end_window',
    'is_hari_besar', 'is_pre_hari_besar',
]
HOLIDAY_INTENSITY_FEATURES = [
    'holiday_intensity', 'days_to_next_holiday',
    'is_holiday_season', 'holiday_x_weekend',
]
TEMPORAL_PEAK_FEATURES = ['is_peak_day']
LAG_ROLL_FEATURES = [
    'demand_lag_1', 'demand_lag_2', 'demand_lag_7', 'demand_lag_14',
    'demand_lag_21', 'demand_lag_28', 'demand_lag_35', 'demand_lag_56', 'demand_lag_84',
    'days_since_last_sale', 'roll_zero_count_14',
    'roll_max_7', 'roll_max_28',
    'roll_mean_7', 'roll_mean_14', 'roll_mean_28', 'roll_mean_56',
    'roll_median_7', 'roll_median_14', 'roll_median_28',
    'roll_std_7', 'roll_std_14', 'roll_std_28', 'roll_std_56',
    'roll_max_56', 'roll_max_84',
    'demand_acceleration_3d',
    'spike_ratio_28', 'spike_ratio_56',
    'pct_change_1', 'pct_change_7',
    'discount_depth_pct', 'price_momentum',
]
ALL_FEATURES = SEASONALITY_FEATURES + HOLIDAY_INTENSITY_FEATURES + TEMPORAL_PEAK_FEATURES + LAG_ROLL_FEATURES

# FIX 1: Feature index map untuk auditability
FEATURE_INDEX_MAP = {i: name for i, name in enumerate(ALL_FEATURES)}
print("[AUDIT] Feature index map (first 5):")
for k in list(FEATURE_INDEX_MAP.keys())[:5]:
    print(f"  f{k} -> {FEATURE_INDEX_MAP[k]}")
print(f"[AUDIT] (total {len(FEATURE_INDEX_MAP)} features)")

TARGET_COL = 'demand_qty'
PRICE_COL = 'avg_price'
DATE_COL = 'date'
print(f"[CONFIG] Total features: {len(ALL_FEATURES)}")



In [ ]:
# ============================================================
# [PREP] Filter + Time-series folds
# ============================================================
MIN_OBS = 60
item_obs = panel_df.groupby(group_cols).size()
panel_df = panel_df[panel_df.set_index(group_cols).index.isin(item_obs[item_obs >= MIN_OBS].index)].copy()
panel_df = panel_df.sort_values(group_cols + ['date']).reset_index(drop=True)
n_items = panel_df[group_cols].drop_duplicates().shape[0]
print(f"[OK] After filter (min_obs={MIN_OBS}): {panel_df.shape}, items: {n_items}")

def time_series_folds(dates, horizon_days=30, n_splits=3, min_train_days=180):
    dates = np.array(sorted(pd.to_datetime(dates).unique()))
    total_days = len(dates)
    splits = []
    for i in range(n_splits):
        val_end_idx = total_days - (n_splits - i - 1) * horizon_days
        val_start_idx = val_end_idx - horizon_days
        train_end_idx = val_start_idx - 1
        if train_end_idx < min_train_days:
            continue
        splits.append((dates[train_end_idx], dates[val_start_idx], dates[val_end_idx - 1]))
    return splits

splits = time_series_folds(panel_df[DATE_COL], horizon_days=30, n_splits=3, min_train_days=180)
print(f"[OK] Folds generated: {len(splits)}")
for i, (tr, vs, ve) in enumerate(splits):
    print(f"  Fold {i+1}: train<={tr.date()} | val={vs.date()} to {ve.date()}")



In [ ]:
# ============================================================
# [METRICS] Evaluation functions
# ============================================================
def calc_cls(y_true, y_pred, avg_price_arr, margin=0.20):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    avg_price = np.asarray(avg_price_arr, dtype=float)
    cls_val = float(np.sum(np.maximum(y_true - y_pred, 0) * avg_price * margin))
    return cls_val

def smape(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, dtype=float), np.asarray(y_pred, dtype=float)
    denom = np.abs(y_true) + np.abs(y_pred)
    mask = denom != 0
    return 0.0 if mask.sum() == 0 else float(np.mean(np.abs(y_true[mask] - y_pred[mask]) / denom[mask]) * 100)

def evaluate_prediction(y_true, y_pred, avg_price_arr, y_naive=None):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2)))
    smape_val = smape(y_true, y_pred)
    y_true_bin = (np.asarray(y_true) > 0).astype(int)
    y_pred_bin = (np.asarray(y_pred) > 0).astype(int)
    f1_zero = f1_score(y_true_bin, y_pred_bin, zero_division=0)
    precision_zero = precision_score(y_true_bin, y_pred_bin, zero_division=0)
    recall_zero = recall_score(y_true_bin, y_pred_bin, zero_division=0)
    cls = calc_cls(y_true, y_pred, avg_price_arr)
    ofr = float(np.minimum(y_true, y_pred).sum() / (y_true.sum() + 1e-8))
    oos_rate = float(np.mean(y_pred < y_true))
    fva = None
    if y_naive is not None:
        baseline_mae = mean_absolute_error(y_true, y_naive)
        if baseline_mae > 0:
            fva = float((baseline_mae - mae) / baseline_mae)
    return {
        'mae': mae, 'rmse': rmse, 'smape': smape_val,
        'f1_zero': f1_zero, 'precision_zero': precision_zero, 'recall_zero': recall_zero,
        'cls': cls, 'ofr': ofr, 'oos_rate': oos_rate, 'fva': fva
    }

print("[OK] Metrics functions loaded")



In [ ]:
# ============================================================
# [CONFIG] Hyperparameters for Decoupled Architecture
# ============================================================
MODEL_PARAMS = {
    'n_estimators': 400, 'max_depth': 5, 'learning_rate': 0.05,
    'subsample': 0.8, 'colsample_bytree': 0.8,
    'device': DEVICE, 'tree_method': 'hist', 'max_bin': 128,
    'random_state': RANDOM_STATE, 'n_jobs': -1,
}

QUANTILE_ALPHA = 0.85  # risk-aware quantile target
USE_LOG_TARGET = True
TOP_SEGMENT_PCT = 0.05

print(f"[CONFIG] Device: {DEVICE}")
print(f"[CONFIG] MODEL_PARAMS: {MODEL_PARAMS}")
print(f"[CONFIG] QUANTILE_ALPHA: {QUANTILE_ALPHA}")
print(f"[CONFIG] USE_LOG_TARGET: {USE_LOG_TARGET}")



In [ ]:
# ============================================================
# [TRAINING] Decoupled 3-Pronged Architecture
# ============================================================
# NOTA: Semua model menerima DataFrame langsung (bukan NumPy)
# sehingga XGBoost menyimpan nama fitur asli untuk feature importance.
# Ini adalah FIX 1 dari anomaly "f34".

def run_decoupled_model(feature_cols, label, panel_df, splits):
    fold_rows = []
    fold_cache = []
    print(f'[TRAIN] 3-Pronged Decoupled | device={DEVICE}')
    for fold_idx, (train_end, val_start, val_end) in enumerate(splits, start=1):
        _t = time.time()
        print(f'\n--- Fold {fold_idx}/{len(splits)} ---')

        train_mask = panel_df[DATE_COL] <= train_end
        val_mask = (panel_df[DATE_COL] >= val_start) & (panel_df[DATE_COL] <= val_end)

        df_tr = panel_df.loc[train_mask].reset_index(drop=True)
        df_vl = panel_df.loc[val_mask].reset_index(drop=True)

        # FIX 1: DataFrame langsung, bukan .to_numpy()
        X_tr = df_tr[feature_cols]
        y_tr = df_tr[TARGET_COL].to_numpy(dtype=np.float32, copy=False)
        X_vl = df_vl[feature_cols]
        y_vl = df_vl[TARGET_COL].to_numpy(dtype=np.float32, copy=False)
        price_vl = df_vl[PRICE_COL].to_numpy(dtype=np.float32, copy=False)

        # Target untuk regressor (hanya non-zero rows)
        nz = y_tr > 0
        X_nz = X_tr.loc[nz]
        y_nz = y_tr[nz]
        y_reg = np.log1p(y_nz) if USE_LOG_TARGET else y_nz

        # ============================================================
        # STEP 1: Honest Baseline (Symmetric Regressor)
        # Objektif: Memprediksi mean demand sejujur mungkin tanpa bias
        # Loss: reg:squarederror = MSE, tidak ada penalty under/over
        # ============================================================
        print(f"  [Fold {fold_idx}] Training Step 1: Honest Baseline...")
        model_mean = xgb.XGBRegressor(
            **MODEL_PARAMS,
            objective='reg:squarederror',
            feature_names=feature_cols,
        )
        model_mean.fit(X_nz, y_reg)
        p_mean = model_mean.predict(X_vl)
        p_mean = np.maximum(np.expm1(p_mean) if USE_LOG_TARGET else p_mean, 0)
        print(f"  [Fold {fold_idx}] Step 1 done")

        # ============================================================
        # STEP 2: Risk-Aware Layer (Quantile Regression / Pinball Loss)
        # Objektif: Memprediksi 85th percentile = batas atas confidence
        # Loss: reg:quantileerror dengan quantile_alpha=QUANTILE_ALPHA
        # Pinball loss = asymmetric piecewise linear:
        #   (y - y_hat) * alpha untuk y > y_hat
        #   (y_hat - y) * (1-alpha) untuk y <= y_hat
        # Matematis berbeda dengan alpha=200 sebelumnya:
        # - alpha=200 static: under-predict dihukum 200x lebih keras
        # - Pinball q=0.85: under dihukum (0.85/0.15=5.67)x lebih keras saja
        # - Quantile adaptif terhadap distribusi data, tanpa heuristik
        # ============================================================
        print(f"  [Fold {fold_idx}] Training Step 2: Quantile Risk-Aware...")
        model_quant = xgb.XGBRegressor(
            **MODEL_PARAMS,
            objective='reg:quantileerror',
            quantile_alpha=QUANTILE_ALPHA,
            feature_names=feature_cols,
        )
        model_quant.fit(X_nz, y_reg)
        p_quant = model_quant.predict(X_vl)
        p_quant = np.maximum(np.expm1(p_quant) if USE_LOG_TARGET else p_quant, 0)
        print(f"  [Fold {fold_idx}] Step 2 done")

        # ============================================================
        # STEP 3: Actuarial Optimization (Dynamic Critical Fractile)
        #   CF = Cu / (Cu + Co)
        #   final = mu + (q85 - mu) * CF
        #
        # Generasi metadata sintetis untuk setiap item (stock_code, country):
        # - margin_ratio: berdasarkan avg_price (High jika di atas median global)
        # - shelf_life: berdasarkan kategori (perishable jika avg_price < threshold)
        #
        # Cu (shortage cost) = margin_ratio * (1 + shelf_life_penalty)
        #   High margin -> Cu lebih tinggi -> CF naik -> rekomendasi naik
        #   Perishable -> Cu lebih rendah -> CF turun -> rekomendasi turun
        # Co (overstock cost) = (1 - margin_ratio) * (1 + spoilage_penalty)
        #   Low margin -> Co lebih tinggi -> CF turun -> rekomendasi turun
        #   Perishable -> spoilage_penalty naik -> Co naik -> CF turun
        # ============================================================
        print(f"  [Fold {fold_idx}] Training Step 3: Actuarial Optimization...")

        # -- Synthetic metadata per item --
        # Agregasi harga rata-rata per item untuk menentukan margin
        item_price = df_tr.groupby(['stock_code', 'country'])[PRICE_COL].mean().reset_index()
        global_median_price = item_price[PRICE_COL].median()
        price_dict = item_price.set_index(['stock_code', 'country'])[PRICE_COL].to_dict()

        vl_keys = list(zip(df_vl['stock_code'].to_numpy(), df_vl['country'].to_numpy()))
        item_price_vl = np.array([price_dict.get(k, global_median_price) for k in vl_keys])

        # margin_ratio: normalized [0.5, 1.5], high if above median
        margin_ratio = np.where(
            item_price_vl > global_median_price,
            1.5,   # High margin items: shortage hurts more
            0.7,   # Low margin items: shortage hurts less
        )

        # shelf_life: perishable if avg_price is low (proxy: cheaper items = staples)
        # Dalam FMCG, barang murah (roti, susu) cenderung perishable
        shelf_life_perishable = (item_price_vl < item_price_vl.mean()).astype(float)

        # Cu = shortage cost
        shortage_penalty = margin_ratio * (1.0 + 0.3 * shelf_life_perishable)
        # High margin perishable: Cu = 1.5 * 1.3 = 1.95
        # Low margin non-perishable: Cu = 0.7 * 1.0 = 0.70

        # Co = overstock cost
        spoilage_penalty = 0.5 * shelf_life_perishable  # spoilage adds cost
        overstock_cost = (1.0 - margin_ratio / margin_ratio.max()) * (1.0 + spoilage_penalty)
        # High margin non-perishable: Co = 0 (overstock aman)
        # Low margin perishable: Co = (1-0.7/1.5)*(1+0.5) = 0.53*1.5 = 0.80

        # Critical Fractile per row
        total_cost = shortage_penalty + overstock_cost + 1e-8
        critical_fractile = shortage_penalty / total_cost
        critical_fractile = np.clip(critical_fractile, 0.2, 0.95)  # safety bounds

        # Final recommendation:
        # CF = 0.95 -> mostly q85 (cover high shortage risk)
        # CF = 0.20 -> mostly mean (avoid overstock)
        final_pred = p_mean + (p_quant - p_mean) * critical_fractile
        final_pred = np.maximum(final_pred, 0)

        print(f"  [Fold {fold_idx}] Step 3 done")
        print(f"    CF range: [{critical_fractile.min():.3f}, {critical_fractile.max():.3f}]")
        print(f"    Mean pred: {p_mean.mean():.1f}, Quant: {p_quant.mean():.1f}, Final: {final_pred.mean():.1f}")

        # ---- Evaluate all 3 outputs ----
        y_naive_v = df_vl['demand_lag_1'].to_numpy(dtype=float, copy=False)
        bm = mean_absolute_error(y_vl, np.nan_to_num(y_naive_v, nan=0.0))
        cls_naive = calc_cls(y_vl, np.maximum(np.nan_to_num(y_naive_v, nan=0.0), 0), price_vl)

        results = {}
        for name, pred in [('mean_only', p_mean), ('quant_only', p_quant), ('actuarial', final_pred)]:
            m = evaluate_prediction(y_vl, pred, price_vl, y_naive=y_naive_v)
            m['method'] = name
            m['fold'] = fold_idx
            m['cls_naive'] = cls_naive
            m['cls_reduction_pct'] = (cls_naive - m['cls']) / cls_naive * 100 if cls_naive > 0 else 0.0
            results[name] = m

        fold_rows.append(results)
        fold_cache.append({
            'fold': fold_idx,
            'y_vl': y_vl, 'price_vl': price_vl,
            'p_mean': p_mean, 'p_quant': p_quant, 'p_final': final_pred,
        })

        # Print fold result
        best_method = min(results.values(), key=lambda x: x['cls'])
        print(f"  [Fold {fold_idx}] Best: {best_method['method']} | "
              f"CLS={best_method['cls']:.0f} (naive={cls_naive:.0f}, -{best_method['cls_reduction_pct']:.1f}%) | "
              f"OFR={best_method['ofr']:.3f} | time={time.time()-_t:.1f}s")

        del df_tr, df_vl, X_tr, y_tr, X_vl, y_vl, price_vl
        del model_mean, model_quant
        gc.collect()

    # Aggregate results
    print("\n[DECOUPLED] Aggregating results across folds...")
    df_rows = []
    for fold_result in fold_rows:
        for method_name, m in fold_result.items():
            row = m.copy()
            row['label'] = label
            df_rows.append(row)
    results_df = pd.DataFrame(df_rows)
    return results_df, fold_cache

print("[OK] run_decoupled_model function defined")



In [ ]:
# ============================================================
# [TRAIN] Execute Decoupled 3-Pronged Architecture
# ============================================================
_t0 = time.time()
print('[TRAIN] Starting Decoupled Quantile + Actuarial Optimization...')
results_df, fold_cache = run_decoupled_model(ALL_FEATURES, 'decoupled_v1', panel_df, splits)
print()
print("[RESULT] Aggregated results by method:")
agg = results_df.groupby('method').agg(
    mae=('mae','mean'), rmse=('rmse','mean'),
    cls=('cls','mean'), ofr=('ofr','mean'),
    smape=('smape','mean'), cls_red=('cls_reduction_pct','mean'),
).round(4)
display(agg)
print(f"\n[TIME] Total training: {time.time()-_t0:.1f}s")



In [ ]:
# ============================================================
# [EVAL] Fold details per method
# ============================================================
print("[EVAL] Per-fold metrics:")
for fold_num in sorted(results_df['fold'].unique()):
    print(f"\n  Fold {fold_num}:")
    fld = results_df[results_df['fold'] == fold_num]
    for _, row in fld.iterrows():
        print(f"    {row['method']:15s} | MAE={row['mae']:8.2f} | CLS={row['cls']:8.0f} | OFR={row['ofr']:.4f} | CLS_red={row['cls_reduction_pct']:.1f}%")



In [ ]:
# ============================================================
# [EVAL] Quantile bucket analysis (actuarial only)
# ============================================================
_t0 = time.time()
print("[EVAL] Quantile bucket evaluation (Actuarial vs Mean vs Quant)...")

bucket_rows = []
for fold_idx, cache in enumerate(fold_cache, start=1):
    y_vl = cache['y_vl']
    price_vl = cache['price_vl']

    for method_name, pred_key in [
        ('mean_only', 'p_mean'), ('quant_only', 'p_quant'), ('actuarial', 'p_final')
    ]:
        pred = cache[pred_key]
        nz_mask = y_vl > 0
        if nz_mask.sum() == 0:
            continue
        nz_true = y_vl[nz_mask]
        nz_pred = pred[nz_mask]
        nz_price = price_vl[nz_mask]
        try:
            buckets = pd.qcut(nz_true, q=[0.0, 0.5, 0.8, 0.95, 1.0],
                              labels=['0-50', '50-80', '80-95', '95-100'], duplicates='drop')
        except ValueError:
            continue
        for bucket in buckets.unique():
            mask = buckets == bucket
            m = evaluate_prediction(nz_true[mask], nz_pred[mask], nz_price[mask])
            m['fold'] = fold_idx
            m['bucket'] = bucket
            m['method'] = method_name
            bucket_rows.append(m)

bdf = pd.DataFrame(bucket_rows)
print("\n[QUANTILE BUCKET] Actuarial vs Mean vs Quant:")
pivot = bdf.groupby(['method', 'bucket'])[['mae', 'cls', 'ofr']].mean().round(2)
display(pivot)
print(f"[TIME] Quantile eval: {time.time()-_t0:.1f}s")



In [ ]:
# ============================================================
# [ANALYSIS] Feature importance (subsample 100k)
# FIX 1: feature_names dipertahankan dari DataFrame
# ============================================================
_t0 = time.time()
nz_mask = panel_df[TARGET_COL] > 0
nz_idx = np.where(nz_mask.to_numpy())[0]
rng = np.random.default_rng(RANDOM_STATE)
samp_idx = rng.choice(nz_idx, size=min(100_000, len(nz_idx)), replace=False)
X_s = panel_df.iloc[samp_idx][ALL_FEATURES]
y_s = np.log1p(panel_df.iloc[samp_idx][TARGET_COL].to_numpy()) if USE_LOG_TARGET else panel_df.iloc[samp_idx][TARGET_COL].to_numpy()

# Train quantile model on subsample (feature_names otomatis dari DataFrame)
ip = xgb.XGBRegressor(
    n_estimators=100, max_depth=6, learning_rate=0.1,
    device=DEVICE, tree_method='hist',
    random_state=RANDOM_STATE, n_jobs=-1,
    objective='reg:quantileerror', quantile_alpha=QUANTILE_ALPHA,
)
ip.fit(X_s, y_s)

# FIX 1: Get_score sekarang mengembalikan nama fitur asli, bukan f0..f51
imp = ip.get_booster().get_score(importance_type='gain')
imp_df = pd.DataFrame.from_dict(imp, orient='index', columns=['gain']).sort_values('gain', ascending=False)
imp_df['gain_pct'] = imp_df['gain'] / imp_df['gain'].sum() * 100

print("[FEATURE IMPORTANCE] Top 15:")
display(imp_df.head(15))

# Export mapping untuk audit
feature_map = pd.DataFrame({
    'index': range(len(ALL_FEATURES)),
    'feature_name': ALL_FEATURES,
})
feature_map.to_json("/kaggle/working/feature_index_map.json", orient="records")
print("[AUDIT] Feature index map saved: /kaggle/working/feature_index_map.json")
print(f"[TIME] Feature importance: {time.time()-_t0:.1f}s")



In [ ]:
# ============================================================
# [VALIDASI] Holiday Intensity Distribution
# Memastikan tidak flat di 10.0 (FIX 2 berhasil)
# ============================================================
hi = panel_df['holiday_intensity']
print("[VALIDASI] Holiday Intensity Distribution:")
print(f"  Count:  {len(hi)}")
print(f"  Mean:   {hi.mean():.4f}")
print(f"  Std:    {hi.std():.4f}")
print(f"  Min:    {hi.min():.4f}")
print(f"  25%:    {hi.quantile(0.25):.4f}")
print(f"  50%:    {hi.quantile(0.50):.4f}")
print(f"  75%:    {hi.quantile(0.75):.4f}")
print(f"  Max:    {hi.max():.4f}")
print(f"  At cap ({HOLIDAY_INTENSITY_CAP}): {(hi >= HOLIDAY_INTENSITY_CAP).sum()} rows ({(hi >= HOLIDAY_INTENSITY_CAP).mean()*100:.2f}%)")
print(f"  Flat di 10.0 (old bug): {(hi == 10.0).sum()} rows")

# Top countries by mean intensity
country_hi = panel_df.groupby('country_code')['holiday_intensity'].agg(['mean', 'std', 'count']).sort_values('mean', ascending=False)
print("\nTop 10 countries by mean holiday_intensity:")
print(country_hi.head(10).to_string())



In [ ]:
# ============================================================
# [VIZ] Performance visualization
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Decoupled Quantile + Actuarial Optimization', fontsize=14, fontweight='bold')

# 1. CLS per fold (actuarial only)
ax = axes[0, 0]
try:
    a_cls = results_df[results_df['method']=='actuarial'].groupby('fold')['cls'].mean()
    m_cls = results_df[results_df['method']=='mean_only'].groupby('fold')['cls'].mean()
    q_cls = results_df[results_df['method']=='quant_only'].groupby('fold')['cls'].mean()
    x = a_cls.index
    ax.plot(x, a_cls.values, 'o-', label='Actuarial', color='#2ca02c', linewidth=2)
    ax.plot(x, m_cls.values, 's--', label='Mean (baseline)', color='#1f77b4')
    ax.plot(x, q_cls.values, '^--', label='Quant (q=0.85)', color='#ff7f0e')
    ax.set_xlabel('Fold'); ax.set_ylabel('CLS')
    ax.set_title('CLS per Fold'); ax.legend(); ax.grid(True, alpha=0.3)
except: ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)

# 2. OFR per fold
ax = axes[0, 1]
try:
    a_ofr = results_df[results_df['method']=='actuarial'].groupby('fold')['ofr'].mean()
    m_ofr = results_df[results_df['method']=='mean_only'].groupby('fold')['ofr'].mean()
    q_ofr = results_df[results_df['method']=='quant_only'].groupby('fold')['ofr'].mean()
    ax.plot(x, a_ofr.values, 'o-', label='Actuarial', color='#2ca02c', linewidth=2)
    ax.plot(x, m_ofr.values, 's--', label='Mean (baseline)', color='#1f77b4')
    ax.plot(x, q_ofr.values, '^--', label='Quant (q=0.85)', color='#ff7f0e')
    ax.axhline(y=0.8, color='red', linestyle=':', label='Target 0.8')
    ax.set_xlabel('Fold'); ax.set_ylabel('OFR')
    ax.set_title('OFR per Fold'); ax.legend(); ax.grid(True, alpha=0.3)
except: ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)

# 3. Aggregated CLS bar
ax = axes[0, 2]
try:
    agg_cls = results_df.groupby('method')['cls'].mean().sort_values()
    colors = ['#2ca02c' if m == 'actuarial' else '#1f77b4' for m in agg_cls.index]
    bars = ax.bar(agg_cls.index, agg_cls.values, color=colors, width=0.5)
    for bar, v in zip(bars, agg_cls.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+200, f'{v:.0f}',
                ha='center', fontsize=9)
    ax.set_ylabel('CLS'); ax.set_title('Mean CLS by Method'); ax.grid(True, alpha=0.3)
except: ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)

# 4. Quantile bucket OFR
ax = axes[1, 0]
try:
    bucket_ofr = bdf.groupby(['method','bucket'])['ofr'].mean().unstack()
    bucket_ofr.T.plot(ax=ax, marker='o', linewidth=2)
    ax.axhline(y=0.8, color='red', linestyle=':', label='Target 0.8')
    ax.set_xlabel('Demand Quantile Bucket'); ax.set_ylabel('OFR')
    ax.set_title('OFR by Demand Quantile Bucket')
    ax.legend(title='Method'); ax.grid(True, alpha=0.3)
except Exception as e:
    ax.text(0.5, 0.5, f'Error: {str(e)[:50]}', ha='center', va='center', transform=ax.transAxes)

# 5. Actuarial: CLS Reduction % per fold
ax = axes[1, 1]
try:
    cls_red = results_df[results_df['method']=='actuarial'].groupby('fold')['cls_reduction_pct'].mean()
    ax.bar(cls_red.index, cls_red.values, color='#2ca02c', width=0.5)
    for i, v in enumerate(cls_red.values):
        ax.text(i+1, v+0.5, f'{v:.1f}%', ha='center')
    ax.set_xlabel('Fold'); ax.set_ylabel('CLS Reduction (%)')
    ax.set_title('CLS Reduction vs Naive (Actuarial)')
    ax.grid(True, alpha=0.3)
except: ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)

# 6. Critical Fractile distribution (fold 1)
ax = axes[1, 2]
try:
    fold0_cache = fold_cache[0]
    mae_f1 = mean_absolute_error(fold0_cache['y_vl'], fold0_cache['p_final'])
    rmse_f1 = float(np.sqrt(np.mean((fold0_cache['y_vl'] - fold0_cache['p_final'])**2)))
    ax.text(0.1, 0.7, f"Fold 1 Performance:\nMAE = {mae_f1:.1f}\nRMSE = {rmse_f1:.1f}", fontsize=11,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    ax.set_title('Sample Performance')
    ax.axis('off')
except: ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)

plt.tight_layout()
plt.show()



In [ ]:
# ============================================================
# [SUMMARY] Final Results
# ============================================================
avg_naive = results_df[results_df['method']=='actuarial']['cls_naive'].mean()
avg_actuarial = results_df[results_df['method']=='actuarial']['cls'].mean()
avg_mean = results_df[results_df['method']=='mean_only']['cls'].mean()
avg_quant = results_df[results_df['method']=='quant_only']['cls'].mean()
cls_red = results_df[results_df['method']=='actuarial']['cls_reduction_pct'].mean()

print()
print('='*70)
print('DECOUPLED QUANTILE + ACTUARIAL OPTIMIZATION -- RINGKASAN')
print('='*70)
print(f'''
Parameter:
  Device        = {DEVICE}
  Quantile      = q = {QUANTILE_ALPHA}
  Fitur         = {len(ALL_FEATURES)}
  FIX 1 (f34)   = feature_names dipertahankan via DataFrame
  FIX 2 (holiday)= clip diangkat ke {HOLIDAY_INTENSITY_CAP}, agregasi per tanggal

CLS (Cost of Lost Sales):
  Naive     = {avg_naive:>8.0f}
  Mean only = {avg_mean:>8.0f}  ({100*(1-avg_mean/avg_naive):>5.1f}% vs naive)
  Quant q{QUANTILE_ALPHA:.2f} = {avg_quant:>8.0f}  ({100*(1-avg_quant/avg_naive):>5.1f}% vs naive)
  Actuarial = {avg_actuarial:>8.0f}  ({cls_red:.1f}% vs naive)

Kesimpulan:
  1. Alpha=200 statis diganti dengan Quantile Regression (Pinball Loss)
     yang adaptif terhadap distribusi data.
  2. Decoupled architecture memisahkan estimasi mean (MSE) dari
     estimasi variance (quantile), sehingga tidak ada bias tunggal.
  3. Dynamic Critical Fractile menyesuaikan rekomendasi per SKU
     berdasarkan margin dan perishability.
''')

print('='*70)
print('Hasil agregat per metode:')
display(agg)

# Feature importance top 5
try:
    top5 = imp_df.head(5)
    print('\nTop 5 Feature Importance (Gain):')
    for idx, row in top5.iterrows():
        print(f'  {idx:25s}: {row["gain_pct"]:.2f}%')
except Exception:
    pass

print()
print('[DONE] Notebook selesai -- Decoupled Quantile + Actuarial Optimization')
print('='*70)



In [ ]:
# ============================================================
# [OPTUNA] Dual-Dimension Hyperparameter Tuning
# ============================================================
# Mencari parameter ML + Aktuaria optimal untuk memenuhi:
#   OFR >= 0.80  |  CLS <= 50,000

try:
    import optuna
except ImportError:
    !pip install optuna -q
    import optuna

def objective(trial):
    # ML Parameters
    lr = trial.suggest_float('learning_rate', 0.01, 0.2, log=True)
    md = trial.suggest_int('max_depth', 3, 7)
    ss = trial.suggest_float('subsample', 0.7, 1.0)

    # Actuarial Parameters
    qt = trial.suggest_float('q_target', 0.85, 0.98)
    smm = trial.suggest_float('shortage_margin_multiplier', 1.0, 3.0)

    # Build model params
    model_params = {
        'n_estimators': MODEL_PARAMS['n_estimators'],
        'max_depth': md,
        'learning_rate': lr,
        'subsample': ss,
        'colsample_bytree': MODEL_PARAMS['colsample_bytree'],
        'device': DEVICE,
        'tree_method': 'hist',
        'max_bin': 128,
        'random_state': RANDOM_STATE,
        'n_jobs': -1,
    }

    cls_scores = []
    ofr_scores = []

    for fold_idx, (train_end, val_start, val_end) in enumerate(splits, start=1):
        train_mask = panel_df[DATE_COL] <= train_end
        val_mask = (panel_df[DATE_COL] >= val_start) & (panel_df[DATE_COL] <= val_end)

        df_tr = panel_df.loc[train_mask].reset_index(drop=True)
        df_vl = panel_df.loc[val_mask].reset_index(drop=True)

        X_tr = df_tr[ALL_FEATURES]
        y_tr = df_tr[TARGET_COL].to_numpy(dtype=np.float32, copy=False)
        X_vl = df_vl[ALL_FEATURES]
        y_vl = df_vl[TARGET_COL].to_numpy(dtype=np.float32, copy=False)
        price_vl = df_vl[PRICE_COL].to_numpy(dtype=np.float32, copy=False)

        nz = y_tr > 0
        X_nz = X_tr.loc[nz]
        y_nz = y_tr[nz]
        y_reg = np.log1p(y_nz) if USE_LOG_TARGET else y_nz

        # Step 1: Honest Baseline (MSE)
        model_mean = xgb.XGBRegressor(
            **model_params,
            objective='reg:squarederror',
            feature_names=ALL_FEATURES,
        )
        model_mean.fit(X_nz, y_reg)
        p_mean = model_mean.predict(X_vl)
        p_mean = np.maximum(np.expm1(p_mean) if USE_LOG_TARGET else p_mean, 0)

        # Step 2: Risk-Aware Quantile (Pinball Loss dengan q_target)
        model_quant = xgb.XGBRegressor(
            **model_params,
            objective='reg:quantileerror',
            quantile_alpha=qt,
            feature_names=ALL_FEATURES,
        )
        model_quant.fit(X_nz, y_reg)
        p_quant = model_quant.predict(X_vl)
        p_quant = np.maximum(np.expm1(p_quant) if USE_LOG_TARGET else p_quant, 0)

        # Step 3: Actuarial Optimization (Dynamic Critical Fractile)
        item_price = df_tr.groupby(['stock_code', 'country'])[PRICE_COL].mean().reset_index()
        global_median_price = item_price[PRICE_COL].median()
        price_dict = item_price.set_index(['stock_code', 'country'])[PRICE_COL].to_dict()

        vl_keys = list(zip(df_vl['stock_code'].to_numpy(), df_vl['country'].to_numpy()))
        item_price_vl = np.array([price_dict.get(k, global_median_price) for k in vl_keys])

        margin_ratio = np.where(item_price_vl > global_median_price, 1.5, 0.7)
        shelf_life_perishable = (item_price_vl < item_price_vl.mean()).astype(float)

        # Cu = shortage cost * shortage_margin_multiplier
        shortage_penalty = margin_ratio * (1.0 + 0.3 * shelf_life_perishable) * smm
        spoilage_penalty = 0.5 * shelf_life_perishable
        overstock_cost = (1.0 - margin_ratio / margin_ratio.max()) * (1.0 + spoilage_penalty)
        total_cost = shortage_penalty + overstock_cost + 1e-8
        critical_fractile = np.clip(shortage_penalty / total_cost, 0.2, 0.95)

        final_pred = p_mean + (p_quant - p_mean) * critical_fractile
        final_pred = np.maximum(final_pred, 0)

        cls_val = calc_cls(y_vl, final_pred, price_vl)
        ofr_val = float(np.minimum(y_vl, final_pred).sum() / (y_vl.sum() + 1e-8))

        cls_scores.append(cls_val)
        ofr_scores.append(ofr_val)

        del df_tr, df_vl, X_tr, y_tr, X_vl, y_vl, price_vl
        del model_mean, model_quant
        gc.collect()

    # Aggregate across folds
    avg_cls = float(np.mean(cls_scores))
    avg_ofr = float(np.mean(ofr_scores))

    # Constrained Optimization: OFR < 0.80 = catastrophic
    if avg_ofr < 0.80:
        return avg_cls + 1_000_000.0

    return avg_cls


# ---- Run Study ----
print("="*70)
print("[OPTUNA] Memulai Dual-Dimension Hyperparameter Tuning")
print(f"  Jumlah trials: 30")
print(f"  Search Space:")
print(f"    ML       : learning_rate [0.01, 0.2] log, max_depth [3,7], subsample [0.7, 1.0]")
print(f"    Aktuaria : q_target [0.85, 0.98], shortage_margin_multiplier [1.0, 3.0]")
print(f"  Constraint: OFR >= 0.80 (penalty 1.000.000 jika gagal)")
print("="*70)

study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)
study.optimize(objective, n_trials=30, show_progress_bar=True)

# ---- Best Parameters ----
print("\n" + "="*70)
print("[OPTUNA] Hasil Hyperparameter Tuning")
print("="*70)
print(f"\nBest trial: #{study.best_trial.number}")
print(f"  Objective (CLS)  = {study.best_trial.value:>12,.0f}")
print(f"  Parameters:")
for key, val in study.best_trial.params.items():
    print(f"    {key:35s} = {val}")
print()

# ---- Verifikasi Hard Thresholds ----
print("="*70)
print("[VERIFIKASI] Evaluasi parameter terbaik pada full dataset...")
print("="*70)

bp = study.best_trial.params
model_params_best = {
    'n_estimators': MODEL_PARAMS['n_estimators'],
    'max_depth': bp['max_depth'],
    'learning_rate': bp['learning_rate'],
    'subsample': bp['subsample'],
    'colsample_bytree': MODEL_PARAMS['colsample_bytree'],
    'device': DEVICE,
    'tree_method': 'hist',
    'max_bin': 128,
    'random_state': RANDOM_STATE,
    'n_jobs': -1,
}
qt_best = bp['q_target']
smm_best = bp['shortage_margin_multiplier']

cls_best = []
ofr_best = []

for fold_idx, (train_end, val_start, val_end) in enumerate(splits, start=1):
    train_mask = panel_df[DATE_COL] <= train_end
    val_mask = (panel_df[DATE_COL] >= val_start) & (panel_df[DATE_COL] <= val_end)

    df_tr = panel_df.loc[train_mask].reset_index(drop=True)
    df_vl = panel_df.loc[val_mask].reset_index(drop=True)

    X_tr = df_tr[ALL_FEATURES]
    y_tr = df_tr[TARGET_COL].to_numpy(dtype=np.float32, copy=False)
    X_vl = df_vl[ALL_FEATURES]
    y_vl = df_vl[TARGET_COL].to_numpy(dtype=np.float32, copy=False)
    price_vl = df_vl[PRICE_COL].to_numpy(dtype=np.float32, copy=False)

    nz = y_tr > 0
    X_nz = X_tr.loc[nz]
    y_nz = y_tr[nz]
    y_reg = np.log1p(y_nz) if USE_LOG_TARGET else y_nz

    model_mean = xgb.XGBRegressor(**model_params_best, objective='reg:squarederror', feature_names=ALL_FEATURES)
    model_mean.fit(X_nz, y_reg)
    p_mean = model_mean.predict(X_vl)
    p_mean = np.maximum(np.expm1(p_mean) if USE_LOG_TARGET else p_mean, 0)

    model_quant = xgb.XGBRegressor(**model_params_best, objective='reg:quantileerror', quantile_alpha=qt_best, feature_names=ALL_FEATURES)
    model_quant.fit(X_nz, y_reg)
    p_quant = model_quant.predict(X_vl)
    p_quant = np.maximum(np.expm1(p_quant) if USE_LOG_TARGET else p_quant, 0)

    item_price = df_tr.groupby(['stock_code', 'country'])[PRICE_COL].mean().reset_index()
    global_median_price = item_price[PRICE_COL].median()
    price_dict = item_price.set_index(['stock_code', 'country'])[PRICE_COL].to_dict()
    vl_keys = list(zip(df_vl['stock_code'].to_numpy(), df_vl['country'].to_numpy()))
    item_price_vl = np.array([price_dict.get(k, global_median_price) for k in vl_keys])

    margin_ratio = np.where(item_price_vl > global_median_price, 1.5, 0.7)
    shelf_life_perishable = (item_price_vl < item_price_vl.mean()).astype(float)

    shortage_penalty = margin_ratio * (1.0 + 0.3 * shelf_life_perishable) * smm_best
    spoilage_penalty = 0.5 * shelf_life_perishable
    overstock_cost = (1.0 - margin_ratio / margin_ratio.max()) * (1.0 + spoilage_penalty)
    total_cost = shortage_penalty + overstock_cost + 1e-8
    critical_fractile = np.clip(shortage_penalty / total_cost, 0.2, 0.95)

    final_pred = p_mean + (p_quant - p_mean) * critical_fractile
    final_pred = np.maximum(final_pred, 0)

    cls_val = calc_cls(y_vl, final_pred, price_vl)
    ofr_val = float(np.minimum(y_vl, final_pred).sum() / (y_vl.sum() + 1e-8))

    cls_best.append(cls_val)
    ofr_best.append(ofr_val)
    print(f"  Fold {fold_idx}: CLS={cls_val:>10,.0f} | OFR={ofr_val:.4f}")

    del df_tr, df_vl, X_tr, y_tr, X_vl, y_vl, price_vl
    del model_mean, model_quant
    gc.collect()

final_cls = float(np.mean(cls_best))
final_ofr = float(np.mean(ofr_best))

print()
print("="*70)
print("HASIL VERIFIKASI HARD THRESHOLDS")
print("="*70)
print(f"  CLS rata-rata : {final_cls:>10,.0f}  (Threshold: <= 50.000)")
print(f"  OFR rata-rata : {final_ofr:>10.4f}  (Threshold: >= 0.8000)")
print()

if final_ofr >= 0.80 and final_cls <= 50000:
    print("  STATUS: LULUS -- Kedua Hard Thresholds terpenuhi!")
else:
    print("  STATUS: GAGAL -- Hard Thresholds belum terpenuhi.")
    if final_ofr < 0.80:
        print(f"    - OFR {final_ofr:.4f} < 0.80: shortage_margin_multiplier perlu dinaikkan")
    if final_cls > 50000:
        print(f"    - CLS {final_cls:,.0f} > 50.000: q_target perlu diturunkan atau n_estimators dikurangi")

print()
print("[DONE] Optuna tuning selesai.")
print("="*70)
